In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath("C://Users//pc//Desktop//Week 3//insurance-risk-analytics//src//"))

In [26]:
from src.data_loader import load_data
from src.modeling import *

print('Functions imported successfully!')
print()

Failed to reload module 'src.modeling' from file 'c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\src\modeling.py'
Traceback (most recent call last):
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\venv\Lib\site-packages\IPython\extensions\autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\venv\Lib\site-packages\IPython\extensions\autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "C:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\src\modeling.py", line 10, in <module>
    from xg

Functions imported successfully!



[autoreload of src.modeling failed: Traceback (most recent call last):
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\venv\Lib\site-packages\IPython\extensions\autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\venv\Lib\site-packages\IPython\extensions\autoreload.py", line 584, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "C:\Users\pc\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "c:\Users\pc\Desktop\Week 3\insurance-risk-analytics\src\modeling.py", line 10, in <module>
    from xgboost import XGBRegressor
ModuleNotFoundError: No module named 'xgboost'
]


In [4]:
# Load the insurance data
df = load_data('C:/Users/pc/Desktop/Week 3/insurance-risk-analytics/data/insurance_data.csv')
df.head()

,customerid,age,gender,province,vehicletype,annualincome,riskscore,annualpremium,deductible,ncd,...,claimed,claimamount,totalpremium,totalclaims,covertype,automake,vehiclemodel,customvalueestimate,zipcode,transactiondate
0,AC-100000,56,Male,Addis Ababa,Sedan,147270,61,2346,500,30,...,False,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10
1,AC-100001,69,Female,Addis Ababa,SUV,74640,57,2334,500,0,...,True,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13
2,AC-100002,46,Male,Oromia,Sedan,70555,42,1697,250,20,...,False,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17
3,AC-100003,32,Female,Somali,Sedan,89398,63,2370,500,20,...,True,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17
4,AC-100004,60,Female,Tigray,SUV,78475,69,2582,500,0,...,False,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10


In [6]:
df = remove_high_missing_columns(df, threshold=0.5)

In [8]:
df = feature_engineering(df)


In [9]:
df.columns

Index(['customerid', 'age', 'gender', 'province', 'vehicletype',
       'annualincome', 'riskscore', 'annualpremium', 'deductible', 'ncd',
       'pastclaims', 'claimed', 'claimamount', 'totalpremium', 'totalclaims',
       'covertype', 'automake', 'vehiclemodel', 'customvalueestimate',
       'zipcode', 'transactiondate'],
      dtype='str')

In [11]:
severity_df = df[df["totalclaims"] > 0].copy()

In [12]:
target_column = "totalclaims"

X = severity_df.drop(columns=[target_column])

y = severity_df[target_column]

In [14]:
numerical_cols, categorical_cols = separate_features(
    X,
    target_column=None
)


In [15]:
preprocessor = create_preprocessor(
    numerical_cols,
    categorical_cols
)


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (1228, 20)
Testing Shape: (307, 20)


In [ ]:
results = train_and_evaluate_models(
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test
)


print("\nFinal Results")
print("=" * 50)

for result in results:
    print(result)

In [ ]:

linear_model = linear_regression_pipeline(
    preprocessor,
    X_train,
    y_train
)

In [ ]:
rf_model = random_forest_pipeline(
    preprocessor,
    X_train,
    y_train
)


In [ ]:
xgb_model = xgboost_pipeline(
    preprocessor,
    X_train,
    y_train
)

In [ ]:
results = evaluate_models(
    linear_model,
    rf_model,
    xgb_model,
    preprocessor,
    X_test,
    y_test
)


print("\nFinal Results")
print("=" * 50)

for result in results:
    print(result)

In [ ]:
best_model = xgb_model

In [ ]:
X_test_processed = preprocessor.transform(X_test)

In [ ]:
feature_names = get_feature_names(
    preprocessor,
    numerical_cols,
    categorical_cols
)


In [ ]:
shap_values, shap_df = compute_shap_values(
    best_model,
    X_test_processed,
    feature_names
)

In [ ]:
plot_shap_summary(
    shap_values,
    X_test_processed,
    feature_names
)

In [ ]:
plot_shap_importance(
    shap_values,
    X_test_processed,
    feature_names
)


In [ ]:
top_features = get_top_features(
    shap_df,
    top_n=10
)

print("\nTop Influential Features")
print("=" * 50)

print(top_features)

In [ ]:
interpretations = generate_business_interpretation(
    top_features
)

print("\nBusiness Interpretations")
print("=" * 50)

for interpretation in interpretations:
    print(interpretation)